In [2]:
import os
from dotenv import load_dotenv

load_dotenv()


from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter

# Updated modern imports from partner packages
from langchain_neo4j import Neo4jVector, Neo4jGraph
from langchain_ollama import OllamaEmbeddings

In [8]:
url = os.getenv("LOCAL_NEO4J_URI")
username = os.getenv("LOCAL_NEO4J_USERNAME")
password = os.getenv("LOCAL_NEO4J_PASSWORD")

if not all([url, username, password]):
    raise ValueError("Local Neo4j credentials are missing from .env")

graph = Neo4jGraph(
    url=url,
    username=username,
    password=password
)

In [9]:
# Define the folder containing PDF files
folder_path = "./data"

# Initialize an embedding model
embeddings = OllamaEmbeddings(model="llama3.2")

# Load all PDF files from the folder
pdf_documents = []
for file_name in os.listdir(folder_path):
    if file_name.endswith(".pdf"):
        file_path = os.path.join(folder_path, file_name)
        loader = PyPDFLoader(file_path)
        pdf_documents.extend(loader.load())
        
# Split the text into chunks
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
docs = text_splitter.split_documents(pdf_documents)

In [11]:
# Connect to Neo4j and save embeddings
db = Neo4jVector.from_documents(
    docs,
    embedding=embeddings,
    graph = graph
)

print("PDF documents have been successfully processed and saved!")

PDF documents have been successfully processed and saved!


In [12]:
#Creates a retriever to fetch the top 5 most similar vectors from the database using similarity search.
vector_retriever = db.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [13]:
from langchain_classic.chains import RetrievalQA
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")

#Sets up a Retrieval-Augmented QA pipeline where the LLM answers queries using relevant information retrieved from the vector database.
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_retriever,
    return_source_documents=False  
)



In [14]:
rag_chain.invoke("who is the Hitman?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': 'who is the Hitman?',
 'result': 'The answer to your question is not explicitly stated in the provided text. However, based on the context and the character\'s name, Agent 47 is a common protagonist in the Hitman series, and it can be inferred that he might be the main character or protagonist of the game. But without further information, it is not possible to confirm who the specific "Hitman" refers to.'}

In [11]:
rag_chain.invoke("When was Blood Money released?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': 'When was Blood Money released?',
 'result': 'Blood Money was released for Microsoft Windows, PlayStation 2, Xbox, and Xbox 360 in May 2006.'}

In [12]:
rag_chain.invoke("when was Hitman first released?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': 'when was Hitman first released?',
 'result': 'Hitman: Codename 47 was initially released exclusively for Windows in 2000.'}

In [13]:
rag_chain.invoke("What is Agent 47's profession?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': "What is Agent 47's profession?",
 'result': 'Agent 47 is a genetically engineered clone assassin.'}

In [14]:
rag_chain.invoke("Is killing targets the only objective?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': 'Is killing targets the only objective?',
 'result': 'No, killing targets is not the only objective. In fact, according to the text, other objectives can include:\n\n* Completing reviews on websites such as Metacritic and GameRankings\n* Submitting entries into awards shows like the BAFTA Games Awards\n* Being recognized by organizations like the National Academy of Video Game Trade Reviewers\n\nAdditionally, the text mentions that in some cases, physical copies of games were sold, suggesting that other objectives may have included sales targets or revenue goals.'}

In [15]:
rag_chain.invoke("Who plays Agent 47 in the 2007 Hitman movie?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=1, offset=0>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 0, 'line': 1, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k RETURN node.`text` AS text, score, node {.*, `text`: Null, `embedding`: Null } AS metadata'


{'query': 'Who plays Agent 47 in the 2007 Hitman movie?',
 'result': 'Timothy Olyphant.'}

In [21]:
url = os.getenv("NEO4J_URI")
username = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database_name = os.getenv("NEO4J_DATABASE")

if not all([url, username, password, database_name]):
    raise ValueError("Neo4j Aura credentials are missing from .env")

graph2 = Neo4jGraph(
    url=url,
    username=username,
    password=password,
    database=database_name
)

graph2.refresh_schema()

print("Successfully connected to Neo4j Aura!")

Successfully connected to Neo4j Aura!


In [22]:
# # 1. Run a quick command to delete the old 384-dimension index
# try:
#     graph2.query("DROP INDEX vector")
#     print("Successfully deleted the old conflicting index")
# except Exception as e:
#     print(f"Index might already be gone or encountered an issue: {e}")

vector_index = Neo4jVector.from_existing_graph(
    OllamaEmbeddings(model="llama3.2"),
    graph=graph2,
    search_type="hybrid",
    node_label='Chunk',
    text_node_properties=["text"],
    embedding_node_property='embedding'
)


In [23]:
#returning similar vector (cosine similarity)
response = vector_index.similarity_search(
    "when was hitman released?"
)
print(response)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

[]


In [24]:
vector_keyword_retriever = vector_index.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [25]:
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama

# Initialize the chat model
llm = ChatOllama(model="llama3.2")

# Set up your chain
vector_keyword_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vector_keyword_retriever,
    return_source_documents=False  
)


In [26]:
vector_keyword_chain.invoke("can you explain briefly about Hitman series?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

{'query': 'can you explain briefly about Hitman series?',
 'result': 'The Hitman series is a popular stealth action-adventure game franchise created by IO Interactive. The series follows the story of Agent 47, a genetically engineered assassin working for various organizations.\n\nIn the games, players control Agent 47 as he completes contracts in different locations around the world, often using stealth and strategy to evade detection. Each game introduces new features, levels, and gameplay mechanics, while maintaining the core elements of the series.\n\nThe series has evolved over time, with the most recent main installment being "Hitman (2016)" also known as "Hitman: Sniper Assassin" and its sequel "Hitman 2".'}

In [29]:
vector_keyword_chain.invoke("Why was Hitman: Absolution criticized by fans?")

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

{'query': 'Why was Hitman: Absolution criticized by fans?',
 'result': "According to the text, Hitman: Absolution was criticized by fans for its departure from the classic Hitman formula. The game's developers stated that they would move away from sandbox levels and incorporate more action-oriented features, such as a cinematic story-driven narrative and increased emphasis on gunplay, to appeal to a broader audience. This change was not well received by longtime fans of the series who were looking for a return to the stealth-based gameplay of earlier titles in the franchise."}

In [28]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [29]:
google_api_key = os.getenv("GOOGLE_API_KEY")

if not google_api_key:
    raise ValueError("GOOGLE_API_KEY is missing from .env")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [30]:
import os
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from typing import List
from langchain_google_genai import ChatGoogleGenerativeAI


# 2. Initialize the modern Gemini model
# llm = ChatGoogleGenerativeAI(model="gemini-pro", temperature=0)
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# 3. Build the text index inside Neo4j (runs instantly)
graph2.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]"
)

# 4. Define the data blueprint for entity matching
class Entities(BaseModel):
    """Identifying information about entities."""
    names: List[str] = Field(
        ...,
        description="All the person, organization, or business entities that appear in the text",
    )

# 5. Formulate the extraction prompt rules
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are extracting organization and person entities from the text.",
    ),
    (
        "human",
        "Use the given format to extract information from the following input: {question}",
    ),
])

# 6. Bind them together into a structured output chain
entity_chain = prompt | llm.with_structured_output(Entities)


In [31]:
entity_chain.invoke({"question": "Who is agent47?"}).names

['agent47']

In [32]:
from langchain_neo4j.vectorstores.neo4j_vector import remove_lucene_chars

graph2.query(
    "CREATE FULLTEXT INDEX entity IF NOT EXISTS FOR (e:__Entity__) ON EACH [e.id]")

def generate_full_text_query(input: str) -> str:
    """
    Generate a full-text search query for a given input string.

    This function constructs a query string suitable for a full-text
    search. It processes the input string by splitting it into words and 
    appending a similarity threshold (~2 changed characters) to each
    word, then combines them using the AND operator. Useful for mapping
    entities from user questions to database values, and allows for some 
    misspelings.
    """
    full_text_query = ""
    words = [el for el in remove_lucene_chars(input).split() if el]
    for word in words[:-1]:
        full_text_query += f" {word}~2 AND"
    full_text_query += f" {words[-1]}~2"
    return full_text_query.strip()

In [33]:
# Fulltext index query
def structured_retriever(question: str) -> str:
    """
    Collects the neighborhood of entities mentioned
    in the question
    """
    result = ""
    entities = entity_chain.invoke({"question": question})
    print(entities.names)
    for entity in entities.names:
        response = graph2.query(
            """CALL db.index.fulltext.queryNodes('entities', $query, 
            {limit:2})
            YIELD node,score
            CALL {
              MATCH (node)-[r:!MENTIONS]->(neighbor)
              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS 
              output
              UNION
              MATCH (node)<-[r:!MENTIONS]-(neighbor)
              RETURN neighbor.id + ' - ' + type(r) + ' -> ' +  node.id AS 
              output
            }
            RETURN output LIMIT 50
            """,
            {"query": generate_full_text_query(entity)},
        )
        # for el in response:
        #     print(el['output'])
        result += "\n".join([el['output'] for el in response if el['output'] is not None])

In [41]:
print(structured_retriever("Who is Hitman?"))

['Hitman']


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=4, column=13, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entities', $query, \n            {limit:2})\n            YIELD node,score\n            CALL {\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS \n              output\n              UNION\n              MATCH (node)<-[r:!MENTION

None


In [42]:
repsonse = vector_index.similarity_search("Who is Hitman?")
print("received data .....")

for el in response:
    print(el.page_content)

Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

received data .....

text: their original standalone releases.[15] Following the release of Blood Money, IO Interactive shifted its focus to developing new franchises, including the action-packed Kane & Lynch series (2007 and 2010) and the more family-friendly Mini Ninjas (2009).[17] Hitman: Absolution was developed by IO Interactive and published by Square Enix, following their acquisition of Eidos Interactive in 2009. It was released worldwide on 20 November 2012 for Microsoft Windows, OS X, PlayStation 3, and Xbox 360.[18][19][20] The developers stated that Absolution, while retaining stealth elements, would move away from the sandbox levels of the first four games and incorporate more action-oriented features, such as a cinematic story-driven narrative and increased emphasis on gunplay, to appeal to a broader audience.[7] A new mechanic called Instinct was introduced, allowing players to track enemies through walls and highlight objects of interest, a feature that would become a st

In [43]:
#This retriever combines both vector and graph retrievers for same question.
def retriever(question: str):
    print(f"Search query: {question}")
    structured_data = structured_retriever(question)
    unstructured_data = [el.page_content for el in vector_index.similarity_search(question)]
    final_data = f"""Structured data:
{structured_data}
Unstructured data:
{"#Document ". join(unstructured_data)}
    """
    print(final_data)
    return final_data

In [46]:
from langchain_ollama import OllamaLLM

llm = OllamaLLM(model="llama3.2")

In [47]:
from langchain_core.runnables import (
    RunnableBranch,
    RunnableLambda,
    RunnableParallel,
    RunnablePassthrough,
)
from langchain_core.output_parsers.string import StrOutputParser

template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

chain = (
    RunnableParallel(
        {
            "context": retriever,
            "question": RunnablePassthrough(),
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)

In [48]:
response = chain.invoke(input="Who is Hitman?")
print("Final Answer")
print(response)

Search query: Who is Hitman?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=4, column=13, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entities', $query, \n            {limit:2})\n            YIELD node,score\n            CALL {\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS \n              output\n              UNION\n              MATCH (node)<-[r:!MENTION

['Hitman']


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Structured data:
None
Unstructured data:

text:  cast to play Agent 47 but was replaced for unknown reasons.[105] The film also stars Dougray Scott, Robert Knepper, Ulrich Thomsen and Michael Offei.[106] In the film, the International Contract Agency is replaced by a similar group called the Organization, which, like the ICA, benefits from ties to various government agencies, is neutral in global matters and morality, and performs missions all over the world. Unlike the game wherein the hitmen are contracted from a range of backgrounds, the Organization instead recruits orphans and trains them from an early age. A sequel was planned but Olyphant stated on the Nerdist podcast that he had no interest in returning for a sequel and only did the original film in order to pay for his new house following the sudden cancellation of Deadwood.[107] On 5 February 2013, it was reported that the film series was being rebooted with the title Hitman: Agent 47, directed by Aleksander Bach. Screenwrite

In [49]:
response = chain.invoke(input="When will next hitman game be released ?")
print("Final Answer")
print(response)

Search query: When will next hitman game be released ?
[]


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

Structured data:
None
Unstructured data:

text: Aggregate review scores Game Metacritic Hitman: Codename 47 (PC) 73[46] Hitman 2: Silent Assassin (GC) 83[47] (PC) 87[48] (PS2) 85[49] (Xbox) 84[50] Hitman: Contracts (PC) 74[51] (PS2) 80[52] (Xbox) 78[53] Hitman: Blood Money (PC) 82[54] (PS2) 83[55] (Xbox) 81[56] (X360) 82[57] Hitman: Absolution (PC) 79[58] (PS3) 83[59] (X360) 79[60] Hitman (PC) 83[61] (PS4) 84[62] (XONE) 85[63] Hitman 2 (PC) 82[64] (PS4) 82[65] (XONE) 84[66] Hitman 3 (PC) 87[67] (PS5) 84[68] (XSX) 87[69] IO Interactive has confirmed that while Hitman 3 concludes the World of Assassination trilogy, it will not be the final game in the franchise.[41][42][43] During December 2025, in an interview conducted by Variety, IOI CEO Hakan Abrak confirmed that once IOI have shared additional information about the new co-op mode, that is being introduced to the World of Assassination trilogy, they will then begin sharing new information around the next Hitman game.[44] At the IO In

In [50]:

response = chain.invoke(input="How has the Hitman franchise evolved since 2000?")
print("Final Answer")
print(response)

Search query: How has the Hitman franchise evolved since 2000?


Received notification from DBMS server: <GqlStatusObject gql_status='01N00', status_description='warn: feature deprecated. CALL subquery without a variable scope clause is deprecated. Use CALL () { ... }', position=<SummaryInputPosition line=4, column=13, offset=119>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 119, 'line': 4, 'column': 13}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entities', $query, \n            {limit:2})\n            YIELD node,score\n            CALL {\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS \n              output\n              UNION\n              MATCH (node)<-[r:!MENTION

['Hitman franchise']


Received notification from DBMS server: <GqlStatusObject gql_status='01N51', status_description='warn: relationship type does not exist. The relationship type `MENTIONS` does not exist in database `16486358`. Verify that the spelling is correct.', position=<SummaryInputPosition line=5, column=32, offset=157>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 157, 'line': 5, 'column': 32}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL db.index.fulltext.queryNodes('entities', $query, \n            {limit:2})\n            YIELD node,score\n            CALL {\n              MATCH (node)-[r:!MENTIONS]->(neighbor)\n              RETURN node.id + ' - ' + type(r) + ' -> ' + neighbor.id AS \n              output\n              

Structured data:
None
Unstructured data:

text: Hitman Logo of the franchise since 2016; mainly used in the World of Assassination trilogy Genre Stealth Developer IO Interactive Publishers IO Interactive (2021– present)[1] Eidos Interactive (2000–2006) Square Enix (2012–2016, 2022)[2] Warner Bros. Games (2018– 2019)[3] Feral Interactive (2023) Creator Jacob Andersen Platforms Microsoft Windows macOS iOS Android GameCube Meta Quest 3 Nintendo Switch Nintendo Switch 2 PlayStation 2 PlayStation 3 PlayStation 4 PlayStation 5 PlayStation Vita PlayStation VR2 Xbox Xbox 360 Xbox One Xbox Series X and S Hitman (franchise) Hitman is a stealth game franchise created by the Danish developer IO Interactive. The player controls the contract killer Agent 47, who travels the world to assassinate various targets who are assigned to him. The first game in the franchise, Hitman: Codename 47, was published by Eidos Interactive for Microsoft Windows in 2000 and introduced many of the gameplay elements tha

In [62]:
import warnings
warnings.filterwarnings("ignore")


def precision_at_k(relevant_count, k):
    if relevant_count < 0 or k < 0:
        raise ValueError("All input values must be non-negative.")
    if k == 0:
        return 0.0
    return relevant_count / k

def recall_at_k(relevant_count, total_relevant):
    if relevant_count < 0 or total_relevant < 0:
        raise ValueError("All input values must be non-negative.")
    if total_relevant == 0:
        return 0.0
    return relevant_count / total_relevant


test_query = "Who published the early Hitman games?"
expected_keywords = ["eidos", "square enix", "warner bros"]

total_relevant = 4 
k_values = [5, 20, 50]


print("=== LIVE RETRIEVER PERFORMANCE LOG ===")
print(f"{'K Value':<10} | {'Precision@K':<12} | {'Recall@K':<12}")
print("-" * 40)

for k in k_values:

    retriever = vector_index.as_retriever(search_kwargs={"k": k})
    retrieved_docs = retriever.invoke(test_query)
    
    relevant_count = 0
    
    for doc in retrieved_docs:
        text_content = doc.page_content.lower()
        
       
        is_relevant = any(keyword in text_content for keyword in expected_keywords)
        if is_relevant:
            relevant_count += 1
            

    if relevant_count > total_relevant:
        relevant_count = total_relevant
        
    # Calculate your final evaluation percentages
    precision = precision_at_k(relevant_count, k)
    recall = recall_at_k(relevant_count, total_relevant)
    
    print(f"{k:<10} | {precision:<12.2%} | {recall:<12.2%}")

=== LIVE RETRIEVER PERFORMANCE LOG ===
K Value    | Precision@K  | Recall@K    
----------------------------------------


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

5          | 60.00%       | 75.00%      


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

20         | 20.00%       | 100.00%     


Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

50         | 8.00%        | 100.00%     


In [34]:
import time
import warnings
from langchain_community.callbacks.manager import get_openai_callback
from langchain_google_genai import ChatGoogleGenerativeAI

warnings.filterwarnings("ignore")

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

def run_observed_rag_pipeline(user_query, vector_index, llm_model):
    print(f"Executing Query: '{user_query}'\n")
    

    start_retrieval = time.perf_counter()
    
    retriever = vector_index.as_retriever(search_kwargs={"k": 5})
    retrieved_docs = retriever.invoke(user_query)
    
    end_retrieval = time.perf_counter()
    retrieval_latency = end_retrieval - start_retrieval
    
    # Format context for the LLM prompt
    context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])

    system_prompt = f"Answer the question based only on the context provided.\nContext:\n{context_text}"
    
    start_generation = time.perf_counter()
    
    with get_openai_callback() as cb:
        response = llm_model.invoke([
            ("system", system_prompt),
            ("human", user_query)
        ])
        
    end_generation = time.perf_counter()
    generation_latency = end_generation - start_generation
    
    # Calculate global totals
    total_latency = retrieval_latency + generation_latency
    
    print(" FINAL LLM RESPONSE ")
    print(response.content, "\n")
    # print("\n" + "="*40)
    print("      RAG OBSERVABILITY DASHBOARD      ")
    # print("="*40)
    print(f"Retriever Latency (Neo4j): {retrieval_latency:.4f} seconds")
    print(f"Generator Latency (LLM):   {generation_latency:.4f} seconds")
    print(f"Total System Latency:      {total_latency:.4f} seconds")
    # print("-" * 40)
    
    prompt_tokens = cb.prompt_tokens if cb.prompt_tokens > 0 else response.response_metadata.get("token_usage", {}).get("prompt_tokens", 0)
    completion_tokens = cb.completion_tokens if cb.completion_tokens > 0 else response.response_metadata.get("token_usage", {}).get("completion_tokens", 0)
    total_tokens = prompt_tokens + completion_tokens
    
    print(f"Prompt Tokens Ingested:    {prompt_tokens}")
    print(f"Completion Tokens Created: {completion_tokens}")
    print(f"Total Tokens Transacted:   {total_tokens}")
    # print("="*40)

run_observed_rag_pipeline(
    user_query="Who published the early Hitman games?", 
    vector_index=vector_index, 
    llm_model=llm
)

Executing Query: 'Who published the early Hitman games?'



Received notification from DBMS server: <GqlStatusObject gql_status='01N01', status_description='warn: feature deprecated with replacement. db.index.vector.queryNodes is deprecated. It is replaced by SEARCH.', position=<SummaryInputPosition line=1, column=11, offset=10>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 1, 'column': 11}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL () { CALL db.index.vector.queryNodes($vector_index_name, $top_k * $effective_search_ratio, $query_vector) YIELD node, score WITH node, score LIMIT $top_k WITH collect({node:node, score:score}) AS nodes, max(score) AS vector_index_max_score UNWIND nodes AS n RETURN n.node AS node, (n.score / vector_index_max_score) AS score UNION CAL

 FINAL LLM RESPONSE 
I am sorry, but the context provided does not contain any information about who published the early Hitman games. 

      RAG OBSERVABILITY DASHBOARD      
Retriever Latency (Neo4j): 0.6561 seconds
Generator Latency (LLM):   1.5186 seconds
Total System Latency:      2.1748 seconds
Prompt Tokens Ingested:    23
Completion Tokens Created: 89
Total Tokens Transacted:   112
